In [51]:
"""
Dataset Statistics Analysis for Machine Learning Paper
Provides comprehensive statistics about the multimodal gesture dataset
"""

import os
import sys
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple
import matplotlib.pyplot as plt
import seaborn as sns

from datautils.midas import MultimodalGestureDataset

In [52]:


from collections import defaultdict
import numpy as np

def compute_dataset_statistics(dataset, name: str = "Dataset") -> dict:
    stats = {
        'name': name,
        'total_samples': len(dataset),
        'num_classes': dataset.num_classes,
        'classes': list(dataset.classes),           # keep your original codes
        'class_map': dict(getattr(dataset, 'class_map', {})),
    }

    class_counts = defaultdict(int)
    class_durations = defaultdict(list)           # frames
    class_durations_seconds = defaultdict(list)

    for sample in dataset.samples:
        g = int(sample['gesture_code'])
        dur_frames = int(sample['end'] - sample['start'])
        dur_secs = dur_frames / dataset.sample_rate
        class_counts[g] += 1
        class_durations[g].append(dur_frames)
        class_durations_seconds[g].append(dur_secs)

    stats['class_counts'] = {k: int(v) for k, v in class_counts.items()}
    stats['class_durations_frames'] = {k: np.array(v) for k, v in class_durations.items()}
    stats['class_durations_seconds'] = {k: np.array(v) for k, v in class_durations_seconds.items()}

    # Duration stats ONLY for classes that actually appear
    duration_stats = {}
    for g, arr in stats['class_durations_seconds'].items():
        if len(arr) == 0: 
            continue
        duration_stats[g] = {
            'count': len(arr),
            'mean_sec': float(np.mean(arr)),
            'std_sec': float(np.std(arr)),
            'min_sec': float(np.min(arr)),
            'max_sec': float(np.max(arr)),
            'median_sec': float(np.median(arr)),
            'q25_sec': float(np.percentile(arr, 25)),
            'q75_sec': float(np.percentile(arr, 75)),
        }
    stats['duration_stats'] = duration_stats

    # Overall across present classes only
    if len(class_durations_seconds) > 0:
        all_durations = np.concatenate([v for v in class_durations_seconds.values() if len(v) > 0])
    else:
        all_durations = np.array([])
    stats['overall'] = {
        'total_windows': len(dataset),
        'mean_duration_sec': float(np.mean(all_durations)) if all_durations.size else 0.0,
        'std_duration_sec': float(np.std(all_durations)) if all_durations.size else 0.0,
        'min_duration_sec': float(np.min(all_durations)) if all_durations.size else 0.0,
        'max_duration_sec': float(np.max(all_durations)) if all_durations.size else 0.0,
        'median_duration_sec': float(np.median(all_durations)) if all_durations.size else 0.0,
        'total_duration_sec': float(np.sum(all_durations)),
        'total_duration_min': float(np.sum(all_durations) / 60.0),
    }

    # Trial list (safe)
    trials = set()
    for s in dataset.samples:
        start_idx = s['start']
        trials.add(str(dataset.df.loc[start_idx, 'trial_id']))
    stats['num_trials'] = len(trials)
    stats['trial_ids'] = sorted(trials)

    stats['modalities'] = list(dataset.modality_cols.keys())
    stats['modality_dims'] = {m: len(cols) for m, cols in dataset.modality_cols.items()}
    return stats


def print_statistics_report(stats: dict, display_map: dict[int, str] | None = None):
    print("=" * 80)
    print(f"DATASET STATISTICS REPORT: {stats['name']}")
    print("=" * 80)

    # Classes actually present (count > 0), sorted by code
    present = sorted([g for g, c in stats['class_counts'].items() if c > 0])
    label = lambda g: display_map.get(g, str(g)) if display_map else str(g)

    print("\n### OVERALL STATISTICS ###")
    print(f"Total number of samples (windows): {stats['total_samples']:,}")
    print(f"Number of gesture classes (present): {len(present)}")
    print(f"Number of trials: {stats['num_trials']}")
    print(f"Gesture classes: {', '.join(label(g) for g in present)}")

    overall = stats['overall']
    print(f"\nTotal dataset duration: {overall['total_duration_min']:.2f} min ({overall['total_duration_sec']:.2f} s)")
    print(f"Mean window duration: {overall['mean_duration_sec']:.3f} ± {overall['std_duration_sec']:.3f} s")
    print(f"Median window duration: {overall['median_duration_sec']:.3f} s")
    print(f"Duration range: [{overall['min_duration_sec']:.3f}, {overall['max_duration_sec']:.3f}] s")

    print("\n### MODALITY INFORMATION ###")
    print(f"Available modalities: {', '.join(stats['modalities'])}")
    for mod, dim in stats['modality_dims'].items():
        print(f"  - {mod}: {dim} features")

    print("\n### PER-CLASS STATISTICS ###")
    print(f"{'Class':<10} {'Count':<8} {'Mean(s)':<10} {'Std(s)':<10} {'Min(s)':<10} {'Max(s)':<10} {'Median(s)':<10}")
    print("-" * 80)
    for g in present:
        ds = stats['duration_stats'][g]
        print(f"{label(g):<10} {ds['count']:<8} {ds['mean_sec']:<10.3f} {ds['std_sec']:<10.3f} "
              f"{ds['min_sec']:<10.3f} {ds['max_sec']:<10.3f} {ds['median_sec']:<10.3f}")

    print("\n### CLASS DISTRIBUTION ###")
    total = sum(stats['class_counts'][g] for g in present)
    print(f"{'Class':<10} {'Count':<8} {'Percentage':<12}")
    print("-" * 40)
    counts = []
    for g in present:
        c = stats['class_counts'][g]
        counts.append(c)
        pct = (c / total) * 100 if total else 0.0
        print(f"{label(g):<10} {c:<8} {pct:>9.2f}%")

    if counts:
        imbalance_ratio = max(counts) / max(1, min(counts))
        cv = (np.std(counts) / np.mean(counts)) if np.mean(counts) > 0 else 0.0
    else:
        imbalance_ratio, cv = 0.0, 0.0
    print(f"\nClass imbalance ratio (max/min): {imbalance_ratio:.2f}")
    print(f"Coefficient of variation: {cv:.3f}")
    print("\n" + "=" * 80)


import os
import matplotlib.pyplot as plt

def create_visualization_plots(stats: dict, save_dir: str = "./dataset_analysis",
                               display_map: dict[int, str] | None = None):
    os.makedirs(save_dir, exist_ok=True)

    # Present classes only
    present = sorted([g for g, c in stats['class_counts'].items() if c > 0])
    if not present:
        print("[warn] No classes with count > 0 to plot.")
        return

    label = lambda g: display_map.get(g, str(g)) if display_map else str(g)
    x_labels = [label(g) for g in present]
    counts = [stats['class_counts'][g] for g in present]

    # 1) Class distribution
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(x_labels, counts, alpha=0.85, edgecolor='black')
    ax.set_xlabel('Gesture Class', fontsize=12)
    ax.set_ylabel('Number of Samples', fontsize=12)
    ax.set_title('Class Distribution', fontsize=14)
    ax.tick_params(axis='x', rotation=45)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f'{int(count)}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/class_distribution.png", bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_dir}/class_distribution.png")

    # 2) Duration boxplot
    fig, ax = plt.subplots(figsize=(12, 6))
    duration_data = [stats['class_durations_seconds'][g] for g in present]
    bp = ax.boxplot(duration_data, labels=x_labels, patch_artist=True, showmeans=True, meanline=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue'); patch.set_alpha(0.7)
    ax.set_xlabel('Gesture Class', fontsize=12)
    ax.set_ylabel('Duration (seconds)', fontsize=12)
    ax.set_title('Duration Distribution per Class', fontsize=14)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/duration_distributions.png", bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_dir}/duration_distributions.png")

    # 3) Combined
    means = [stats['duration_stats'][g]['mean_sec'] for g in present]
    stds = [stats['duration_stats'][g]['std_sec'] for g in present]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.bar(x_labels, counts, alpha=0.85, edgecolor='black')
    ax1.set_xlabel('Gesture Class', fontsize=11)
    ax1.set_ylabel('Number of Samples', fontsize=11)
    ax1.set_title('Sample Count per Class', fontsize=12)
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3, axis='y')

    ax2.bar(x_labels, means, yerr=stds, alpha=0.85, edgecolor='black', capsize=5)
    ax2.set_xlabel('Gesture Class', fontsize=11)
    ax2.set_ylabel('Mean Duration (seconds)', fontsize=11)
    ax2.set_title('Mean Duration per Class (±1 std)', fontsize=12)
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f"{save_dir}/combined_stats.png", bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_dir}/combined_stats.png")

    # 4) Overall duration histogram
    all_durations = np.concatenate([stats['class_durations_seconds'][g] for g in present])
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(all_durations, bins=30, alpha=0.7, edgecolor='black')
    ax.axvline(np.mean(all_durations), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(all_durations):.2f}s')
    ax.axvline(np.median(all_durations), color='blue', linestyle='--', linewidth=2, label=f'Median: {np.median(all_durations):.2f}s')
    ax.set_xlabel('Duration (seconds)', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title('Overall Duration Distribution', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/duration_histogram.png", bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_dir}/duration_histogram.png")


def export_stats_to_csv(stats: Dict, save_dir: str = "./dataset_analysis"):
    """
    Export statistics to CSV files for easy inclusion in papers/reports.
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # Per-class statistics table
    rows = []
    for gesture in stats['classes']:
        ds = stats['duration_stats'][int(gesture)]
        rows.append({
            'Class': gesture,
            'Count': ds['count'],
            'Mean_Duration_sec': ds['mean_sec'],
            'Std_Duration_sec': ds['std_sec'],
            'Min_Duration_sec': ds['min_sec'],
            'Max_Duration_sec': ds['max_sec'],
            'Median_Duration_sec': ds['median_sec'],
            'Q25_Duration_sec': ds['q25_sec'],
            'Q75_Duration_sec': ds['q75_sec'],
            'Percentage': (ds['count'] / stats['total_samples']) * 100,
        })
    
    df = pd.DataFrame(rows)
    csv_path = f"{save_dir}/class_statistics.csv"
    df.to_csv(csv_path, index=False, float_format='%.3f')
    print(f"Saved: {csv_path}")
    
    # Overall statistics
    overall_df = pd.DataFrame([stats['overall']])
    overall_csv = f"{save_dir}/overall_statistics.csv"
    overall_df.to_csv(overall_csv, index=False, float_format='%.3f')
    print(f"Saved: {overall_csv}")


TypeError: unsupported operand type(s) for |: 'types.GenericAlias' and 'NoneType'

In [ ]:


def main():
    """
    Main analysis script
    """
    # Configuration
    # ROOT_DIR = "/standard/UVA-DSA/MIDAS/Organized/final_data" # peg transfer
    ROOT_DIR = "/standard/UVA-DSA/MIDAS/Organized/Bootcamp/SuturingV2/Processed/"
    
    # List your CSV files here
    # csv_files = [
    #     f"{ROOT_DIR}/t1/synched_data/final_annotation_t1.csv",
    #     f"{ROOT_DIR}/t2/synched_data/final_annotation_t2.csv",
    #     f"{ROOT_DIR}/t3/synched_data/final_annotation_t3.csv",
    #     f"{ROOT_DIR}/t4/synched_data/final_annotation_t4.csv",
    #     f"{ROOT_DIR}/t5/synched_data/final_annotation_t5.csv",
    #     f"{ROOT_DIR}/t6/synched_data/final_annotation_t6.csv",
    #     f"{ROOT_DIR}/t7/synched_data/final_annotation_t7.csv"
    #     # Add more as needed
    # ]

    trials = [
    "S105_T1",
    "S106_T1",
    "S106_T2",
    "S112_T1",
    "S112_T2",
    "S116_T1",
    "S116_T2",
    "S116_T4",
    "S116_T5",
    "S118_T1",
    "S200_T1",
    "S201_T1",
    "S201_T2",
    "S202_T1",
    "S203_T1",
    "S204_T1",
    "S209_T2",
    "S210_T1",
    "S214_T1",
    "S214_T4",
    "S214_T6",
    "S215_T3",
    "S215_T4",
    "S217_T2",
    "S217_T3",
    "S217_T4",
    "S218_T1",
    "S219_T1" ]

    csv_files = [f"{ROOT_DIR}/{trial}/synched_data/final_annotation_{trial}.csv" for trial in trials]
    print(f"Using {len(csv_files)} CSV files for analysis.")
    print("Loading dataset...")
    
    # Create dataset with your desired configuration
    dataset = MultimodalGestureDataset(
        csv_paths=csv_files,
        clip_len=-1,              # Your windowing configuration
        step=1,                   # Your step size
        sample_rate=30,           # Your sampling rate
        ignore_clutch=False,
        clutch_pressed_value=0,
        
        # include_modalities=["trakstar", "sw_left", "sw_right", "console"],
        include_modalities=["trakstar"],
        modality_selections={
            "trakstar": [
          "trakstar_sensor_0_azimuth","trakstar_sensor_1_azimuth","trakstar_sensor_2_azimuth","trakstar_sensor_3_azimuth",
    "trakstar_sensor_0_elevation","trakstar_sensor_1_elevation","trakstar_sensor_2_elevation","trakstar_sensor_3_elevation",
    "trakstar_sensor_0_roll","trakstar_sensor_1_roll","trakstar_sensor_2_roll","trakstar_sensor_3_roll",
    "trakstar_sensor_0_x_transformed","trakstar_sensor_0_y_transformed","trakstar_sensor_0_z_transformed",
      "trakstar_sensor_1_x_transformed","trakstar_sensor_1_y_transformed","trakstar_sensor_1_z_transformed",
      "trakstar_sensor_2_x_transformed","trakstar_sensor_2_y_transformed","trakstar_sensor_2_z_transformed",
       "trakstar_sensor_3_x_transformed","trakstar_sensor_3_y_transformed","trakstar_sensor_3_z_transformed"
            ],
        },
        normalize=True,
        classes_to_allow={  
            8: "Make C Loop",
            11: "Orient Needle",
            16: "Pull Needle out of Tissue",
            17: "Pull Suture",
            18: "Push Needle Through Tissue",
            20: "Reach Suture",
            22: "Square Knot and Cinch",
            23: "Target Needle"
        }
    )
    
    print(f"Dataset loaded: {len(dataset)} samples\n")
    
    # Compute statistics
    print("Computing statistics...")
    stats = compute_dataset_statistics(dataset, name="Multimodal Gesture Dataset")
    
    display_map = {
        8: "G1", 11: "G2", 16: "G3", 17: "G4",
        18: "G5", 20: "G6", 22: "G7", 23: "G8",
    }

    stats = compute_dataset_statistics(dataset, name="Multimodal Gesture Dataset")
    print_statistics_report(stats, display_map=display_map)
    create_visualization_plots(stats, save_dir="./dataset_analysis", display_map=display_map)

        
    # Create visualizations
    print("\nCreating visualizations...")
    save_dir = "./dataset_analysis"
    create_visualization_plots(stats, save_dir=save_dir)
    
    # Export to CSV
    print("\nExporting statistics to CSV...")
    export_stats_to_csv(stats, save_dir=save_dir)
    
    print(f"\n✓ Analysis complete! Results saved to: {save_dir}/")
    print("\nFiles generated:")
    print("  - class_statistics.csv")
    print("  - overall_statistics.csv")
    print("  - class_distribution.png")
    print("  - duration_distributions.png")
    print("  - combined_stats.png")
    print("  - duration_histogram.png")

main()
